In [ ]:
##Linear probe
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "microsoft/phi-2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    output_hidden_states=True,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
).to(device)

model.eval()


def extract_hidden_states(texts, layer_id):
    representations = []
    for text in texts:
        inputs = tokenizer(text, return_tensors="pt", truncation=True).to(device)
        with torch.no_grad():
            outputs = model(**inputs)
        hidden_states = outputs.hidden_states[layer_id]
        pooled = hidden_states.mean(dim=1)
        representations.append(pooled.squeeze(0).cpu())
    return torch.stack(representations)


class LinearProbe(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, 2)

    def forward(self, x):
        return self.linear(x)


def train_probe(representations, labels, epochs=10, lr=1e-3):
    dataset = TensorDataset(representations, labels)
    loader = DataLoader(dataset, batch_size=32, shuffle=True)

    probe = LinearProbe(representations.shape[1]).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(probe.parameters(), lr=lr)

    for _ in range(epochs):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = probe(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

    return probe


def evaluate_probe(probe, representations, labels):
    probe.eval()
    with torch.no_grad():
        logits = probe(representations.to(device))
        preds = torch.argmax(logits, dim=1).cpu()
    accuracy = (preds == labels).float().mean().item()
    return accuracy


def find_most_biased_layer(texts, labels, num_layers):
    best_acc = 0
    best_layer = 0
    best_probe = None

    for layer in range(1, num_layers):
        reps = extract_hidden_states(texts, layer)
        probe = train_probe(reps, labels)
        acc = evaluate_probe(probe, reps, labels)

        if acc > best_acc:
            best_acc = acc
            best_layer = layer
            best_probe = probe

    return best_layer, best_probe


def remove_bias_component(representation, probe):
    w = probe.linear.weight[1] - probe.linear.weight[0]
    w = w.detach().cpu()
    projection = (representation @ w) / (w @ w)
    debiased = representation - projection.unsqueeze(1) * w
    return debiased


def debias_text(text, layer_id, probe):
    inputs = tokenizer(text, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    hidden_states = outputs.hidden_states[layer_id]
    pooled = hidden_states.mean(dim=1).cpu()

    debiased_rep = remove_bias_component(pooled, probe)

    return debiased_rep